# Employee Attrition Prediction - Data Preprocessing

This notebook handles data cleaning, encoding, and scaling.

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import sys
sys.path.append('../src')
from data_preprocessing import DataPreprocessor
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/raw_data.csv')
print(f"Original shape: {df.shape}")

## 2. Handle Missing Values and Duplicates

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Remove duplicates
df = df.drop_duplicates()
print(f"\nShape after removing duplicates: {df.shape}")

# Fill missing numeric values with mean
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

print(f"Missing values after filling:")
print(df.isnull().sum())

## 3. Encode Categorical Variables

In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

# Label encode categorical variables
label_encoders = {}
df_encoded = df.copy()

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"\n{col} encoding:")
    for i, class_ in enumerate(le.classes_):
        print(f"  {class_}: {i}")

print(f"\nEncoded data shape: {df_encoded.shape}")

## 4. Scale Numeric Features

In [ ]:
# Get numeric columns
numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns.tolist()

# Create scaler
scaler = StandardScaler()
df_scaled = df_encoded.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

print("Scaled data statistics:")
print(df_scaled[numeric_cols].describe())

## 5. Split into Train and Test Sets

In [ ]:
# Separate features and target
if 'Attrition' in df_scaled.columns:
    X = df_scaled.drop('Attrition', axis=1)
    y = df_scaled['Attrition']
else:
    print("Warning: 'Attrition' column not found in data")
    X = df_scaled
    y = None

if y is not None:
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Train set size: {X_train.shape}")
    print(f"Test set size: {X_test.shape}")
    print(f"\nTrain target distribution:")
    print(y_train.value_counts())
    print(f"\nTest target distribution:")
    print(y_test.value_counts())

## 6. Save Preprocessed Data

In [ ]:
# Save cleaned data
df.to_csv('../data/cleaned_data.csv', index=False)
df_scaled.to_csv('../data/preprocessed_data.csv', index=False)

# Save train/test split
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

print("Data saved successfully!")